# Modèle de langage et génération de séquence

Dans ces travaux pratiques, nous allons étudier les différentes manières de décoder du texte à partir d'un modèle entraîné. Cette étape est nécessaire pour beaucoup de tâches de traitement du langage : traduction, question/réponse, génération…

Nous prendrons l'exemple de la génération de texte avec GPT-2, un transformer de type *décodeur seul* : les grands modèles de langage actuels (GPT-4, Llama, Mistral…) reposent sur le même principe et se décodent avec les mêmes méthodes.

Nous partirons de la méthode la plus simple pour progresser jusqu'aux méthodes utilisées en pratique, en codant nous-mêmes le choix du token suivant. Cela vous permettra de bien comprendre le rôle de la softmax en sortie des modèles de langage. Pour finir, nous retrouverons tous ces réglages dans la méthode `generate` de la bibliothèque `transformers`.

Un [excellent article](https://huggingface.co/blog/how-to-generate) de Hugging Face peut servir de ressource complémentaire à ce notebook.

Ce TP n'a pas besoin de carte graphique : GPT-2 small tourne sur CPU, et chaque test sur les 7 prompts du TP y prend moins d'une minute. Il utilise le GPU s'il y en a un.

**Attention** : ces modèles ont été entraînés sur des textes du web, sans filtrage. Les textes générés, surtout par échantillonnage, peuvent être incohérents, faux ou choquants.

## Installation

La bibliothèque [`transformers`](https://github.com/huggingface/transformers) de [Hugging Face](https://huggingface.co/) met à disposition un grand nombre de modèles de la famille des transformers. Elle est préinstallée sur Colab ; la cellule suivante l'installe ailleurs (Codespace, poste personnel).

In [ ]:
!pip install -q transformers

## Choix de la langue

Nous allons voir comment générer du texte en anglais mais aussi en français, en chargeant des modèles pré-entraînés sur des corpus différents. La cellule suivante permet de changer la langue utilisée pour le reste du TP.

In [ ]:
langue = "fr"
# langue = "en"

## Import d'un modèle entraîné

En plus de mettre à disposition le code source de nombreuses architectures de transformers, la bibliothèque `transformers` permet de récupérer des [modèles entraînés](https://huggingface.co/models) depuis le Hub Hugging Face. La méthode `from_pretrained` d'`AutoModelForCausalLM` récupère l'architecture et les poids d'un modèle qui prédit le token suivant (*causal language model*), celle d'`AutoTokenizer` son tokeniseur.

- En anglais, `"gpt2"` est le GPT-2 small d'OpenAI (2019).
- En français, [BelGPT-2](https://huggingface.co/antoinelouis/belgpt2) est un GPT-2 small entraîné sur un corpus français.

Les modèles PyTorch sont des `nn.Module` : on les déplace sur le GPU avec `.to(device)`, et on les passe en mode évaluation avec `.eval()` (le dropout est alors désactivé).

Les textes que voit un GPT-2 pendant son entraînement sont séparés par un symbole spécial de début et de fin de texte : `<|endoftext|>` pour GPT-2, `</s>` pour BelGPT-2. La cellule garde son index dans `id_token_debut`.

In [ ]:
import functools
import statistics

import torch
import tqdm.auto
import transformers

# Mêmes tirages aléatoires à chaque exécution du notebook
torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calculs sur : {device}")

nom_pretraine = "antoinelouis/belgpt2" if langue == "fr" else "gpt2"

modele = transformers.AutoModelForCausalLM.from_pretrained(nom_pretraine)
modele = modele.to(device).eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(nom_pretraine)

# Symbole de début et de fin de texte
id_token_debut = tokenizer.convert_tokens_to_ids(
  "</s>" if langue == "fr" else "<|endoftext|>"
)

## Inspection du modèle

Après la récupération d'un modèle pré-entraîné, il est toujours bon de vérifier son architecture et ses caractéristiques. Pour cela, on peut afficher sa configuration et le modèle lui-même (un `nn.Module` s'affiche avec toutes ses couches).

- *Combien le modèle a-t-il de couches de transformer ?*
- *Quels sont ses mécanismes de régularisation ?*
- *De stabilisation de l'apprentissage ?*
- *Combien de paramètres possède-t-il ?*
- *La configuration donne l'index du symbole de début de texte dans `bos_token_id`. Est-ce bien `id_token_debut` ? Affichez le token qui correspond à chacun avec `tokenizer.convert_ids_to_tokens`, en anglais puis en français.*

In [ ]:
print(modele.config)
print(modele)

*Votre réponse à compléter ici.*
-
-
-
-
-

### Solution

In [ ]:
nb_parametres = sum(p.numel() for p in modele.parameters())
print(f"{nb_parametres / 1e6:.0f} millions de paramètres")
print(f"Couches : {modele.config.n_layer}")
for nom, index in [
  ("bos_token_id", modele.config.bos_token_id),
  ("id_token_debut", id_token_debut),
]:
  print(f"{nom} : {index} = {tokenizer.convert_ids_to_tokens(index)!r}")

- Il possède 12 couches de transformer (`n_layer`, ou les 12 `GPT2Block` affichés).
- Son principal mécanisme de régularisation est le dropout (`attn_pdrop`, `resid_pdrop`, `embd_pdrop`).
- Ses mécanismes de stabilisation de l'apprentissage sont la normalisation de couche (`LayerNorm`, `ln_1`, `ln_2`) et les connexions résiduelles.
- Il a environ 124 millions de paramètres. La couche de sortie `lm_head` n'en ajoute aucun : elle réutilise la matrice d'embedding des tokens de l'entrée (`wte`), et `parameters()` ne la compte qu'une fois.
- Pour GPT-2, `bos_token_id` vaut 50256, `<|endoftext|>` : c'est bien `id_token_debut`. Pour BelGPT-2, `bos_token_id` vaut aussi 50256, mais ce token est `ĠArcade` dans son vocabulaire : sa configuration a gardé la valeur de GPT-2, alors que son symbole de début et de fin de texte est `</s>`, d'index 2. Un modèle pré-entraîné n'est pas toujours cohérent avec son tokeniseur : vérifiez ses symboles spéciaux avant de vous en servir.

## Le mécanisme de génération

Pour générer du texte depuis un modèle entraîné, on procède de manière itérative : on génère les tokens un par un, en commençant par un texte vide ou par une phrase que l'on souhaite compléter.

Pour générer un token, on donne en entrée du réseau les tokens générés jusque-là (sous forme d'indices dans le vocabulaire). Le réseau produit alors un score par token du vocabulaire. On choisit l'un de ces tokens, on l'ajoute à ce qui a déjà été généré, puis on recommence pour générer le token suivant.

Plus formellement, on essaye de maximiser $P(\text{Texte}| \text{Initial})$ avec la décomposition suivante :

$$P(\text{Texte}| \text{Initial}) = \prod_{i=1}^{\text{Taille du texte}} P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$$

Tout l'enjeu du décodage est de trouver le texte qui maximise ce produit : c'est la meilleure proposition du modèle. Cette valeur est impossible à calculer exactement, on a donc recours à des heuristiques pour l'approcher.

C'est le **masque causal** du transformer qui rend cette décomposition possible : dans chaque couche d'attention, la position $i$ ne peut regarder que les positions $j \le i$. Pendant l'entraînement, le modèle apprend ainsi à prédire chaque token à partir des seuls tokens précédents : exactement ce dont la génération a besoin.

## Tokenisation

Le tokeniseur récupéré avec le modèle transforme une phrase en indices de vocabulaire (des *sous-mots*) que le modèle comprend. Dans l'affichage des tokens, `Ġ` marque un token qui commence par une espace.

Nous utiliserons dans la suite du TP les fonctions `encode` et `decode` suivantes. `encode` ajoute le symbole de début de texte, et renvoie un tenseur de forme `(1, nombre de tokens)` sur le bon device.

BelGPT-2 a été entraîné sur des textes où la ponctuation est séparée des mots par des espaces (« j' ai », « fin . ») : ses générations en gardent la trace. Il colle aussi parfois deux phrases sans espace entre elles (« soir.Les ») : un défaut du modèle (des fins de ligne retirées de son corpus, sans doute), pas de la bibliothèque.

In [ ]:
def encode(phrase: str) -> torch.Tensor:
  """Indices des tokens, après le symbole de début : forme (1, nombre de tokens)."""
  ids = tokenizer.encode(phrase, add_special_tokens=False, return_tensors="pt")
  debut = torch.tensor([[id_token_debut]])
  return torch.cat([debut, ids], dim=1).to(device)


def decode(ids: torch.Tensor) -> str:
  """Texte des tokens, sans le symbole de début."""
  return tokenizer.decode(ids.squeeze(0)[1:])


exemple = encode(
  "Je pense, donc je suis" if langue == "fr" else "I think, therefore I am"
)
print(exemple)
print(tokenizer.convert_ids_to_tokens(exemple[0]))

## Étude des sorties du modèle pré-entraîné

La première étape pour décoder des phrases depuis un modèle appris est de faire une passe forward du modèle pour récupérer ses prédictions pour le prochain token. Étudions son comportement.

Comme tout module PyTorch, on calcule la passe forward avec `modele(ids_tokens)`. Le résultat est un objet dont les attributs principaux sont :

- `logits` : les scores du modèle, avant softmax ;
- `past_key_values` : un cache des clés et valeurs d'attention déjà calculées.

*Appliquez le modèle à diverses séquences, par exemple :*

- *« » (le texte vide)*
- *« Je pense, donc je suis » ou « I think, therefore I am » en fonction de la langue du modèle*

*Quelle est la forme des `logits` ? À quoi correspond chaque dimension ?*

*Désactivez le calcul des gradients : nous n'entraînons pas le modèle.*

**Documentation utile**

- [`GPT2LMHeadModel`](https://huggingface.co/docs/transformers/model_doc/gpt2#transformers.GPT2LMHeadModel)
- [`torch.no_grad`](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels à utiliser :

- `encode(phrase)` donne les `ids_tokens`, de forme `(1, nombre de tokens)`
- `with torch.no_grad():` : les calculs du bloc ne gardent pas de quoi calculer les gradients
- `sortie = modele(ids_tokens)`, puis `sortie.logits.shape` donne la forme des logits
- Résultat attendu pour le texte vide : `(1, 1, 50257)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def decrit_formes(phrase: str) -> None:
  with torch.no_grad():
    sortie = modele(...)
  print(f"Forme des logits pour '{phrase}' : {tuple(...)}")


decrit_formes("")
decrit_formes("Je pense, donc je suis" if langue == "fr" else "I think, therefore I am")
```

### Solution

In [ ]:
def decrit_formes(phrase: str) -> None:
  with torch.no_grad():
    sortie = modele(encode(phrase))
  print(f"Forme des logits pour '{phrase}' : {tuple(sortie.logits.shape)}")


decrit_formes("")
decrit_formes("Je pense, donc je suis" if langue == "fr" else "I think, therefore I am")

Les logits ont la forme `(batch, nombre de tokens, taille du vocabulaire)` : pour chaque position de l'entrée, le modèle donne un score à chacun des 50 257 tokens du vocabulaire pour la position **suivante**. Le texte vide a une position, celle du symbole de début qu'`encode` ajoute. Pour générer, seule la dernière position nous intéresse.

## Fonction `forward` adaptée aux besoins du décodage

Nous utiliserons la fonction `forward` suivante, qui rend un résultat adapté au décodage : seule la dernière position nous intéresse (la prédiction du token suivant).

Elle utilise le cache `past_key_values` : il évite de recalculer, à chaque nouveau token, les clés et valeurs d'attention de tous les tokens précédents. Quand le cache est donné, il ne faut passer au modèle **que les nouveaux tokens**.

In [ ]:
@torch.no_grad()
def forward(ids: torch.Tensor, cache=None) -> tuple[torch.Tensor, object]:
  """Logits du token suivant, de forme (1, taille du vocabulaire), et le cache."""
  sortie = modele(ids, past_key_values=cache, use_cache=True)
  return sortie.logits[:, -1, :], sortie.past_key_values

Il faut donc utiliser au choix :

- `forward(tous_les_tokens)`
- `forward(dernier_token, cache)`

où `tous_les_tokens` serait par exemple une séquence de 7 tokens de forme `(1, 7)` là où `dernier_token` serait de forme `(1, 1)`. Les logits renvoyés ont la forme `(1, taille du vocabulaire)` : la dimension du batch, de taille 1, est conservée.

*Le modèle calcule des logits pour chaque position, mais la génération n'utilise que ceux de la dernière. À quoi servent les autres, pendant l'entraînement ? Et pourquoi les logits de la position $i$ ne dépendent-ils pas des tokens qui la suivent ?*

*Votre réponse à compléter ici.*

### Solution

Pendant l'entraînement, chaque position prédit le token qui la suit : sur un texte de $n$ tokens, une seule passe forward donne $n$ prédictions, toutes comparées à la fois aux vrais tokens suivants (une entropie croisée par position, moyennée).

C'est le masque causal qui le permet : la position $i$ ne voit que les tokens jusqu'à $i$, sans quoi elle lirait directement le token qu'elle doit prédire. Le modèle apprend ainsi dans les conditions de la génération, où les tokens suivants n'existent pas encore.

## Boucle de décodage

Une boucle de décodage suit toujours le même schéma :

1. Passe forward sur ce qui a été produit jusqu'ici (entrées et sorties précédentes ; grâce au cache, seulement le dernier token après le premier pas)
2. Choix de l'indice du token suivant
3. Répétition de 1. et 2. jusqu'à ce qu'un critère d'arrêt soit satisfait (nous utiliserons seulement le nombre de tokens produits dans ce TP)

L'étape 2 est celle où tout se joue. Le code suivant implémente tout le reste, afin de pouvoir nous concentrer sur l'étape 2 : chaque méthode de décodage sera une fonction qui reçoit les logits du dernier token (forme `(1, taille du vocabulaire)`) et renvoie l'indice du token choisi (un tenseur de forme `()`). Décorée par `@boucle_decodage`, elle devient une fonction `(prompt, nombre de tokens à générer, paramètres de la méthode)` qui renvoie le texte généré.

In [ ]:
def boucle_decodage(fonction_pas):

  @functools.wraps(fonction_pas)
  def enveloppe(prompt: str, longueur: int, *args_pas, **kwargs_pas) -> str:
    decodes = [encode(prompt)]
    cache = None
    for _ in tqdm.auto.trange(longueur, desc="Tokens", leave=False):
      logits, cache = forward(decodes[-1], cache)
      index = fonction_pas(logits, *args_pas, **kwargs_pas)
      decodes.append(index.reshape(1, 1))
    return decode(torch.cat(decodes, dim=1))

  return enveloppe

## Décodage glouton (*greedy*)

Le décodage glouton est la méthode la plus simple pour décoder du texte depuis un modèle appris : à chaque étape, on prend le token de plus forte probabilité.

Dans l'équation :

$$P(\text{Texte}|\text{Initial}) = \prod_{i=1}^{\text{Taille du texte}} P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$$

le décodage glouton choisit $\text{Token}_i$ pour maximiser $P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$. Or, il faut souvent choisir un token moins probable pour ensuite atteindre des tokens très probables.

L'exemple suivant est tiré de l'article de Hugging Face sur la génération de texte :

![Greedy decoding](https://huggingface.co/blog/assets/02_how-to-generate/greedy_search.png)

Après « The », on choisit « nice » parce que c'est le mot le plus probable, alors que le texte complet « The nice woman » a une probabilité (0,20) inférieure à celle qu'on aurait obtenue en choisissant « dog » (« The dog has », 0,36).

*Codez la fonction `glouton(logits)` qui prend en entrée les logits du modèle et renvoie l'indice du token choisi, un tenseur de forme `()`. Elle utilisera `torch.argmax`. Générez 10 tokens après « Je suis allé au » (« I went to the » en anglais).*

**Documentation utile**

- [`torch.argmax`](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

In [ ]:
@boucle_decodage
def glouton(logits: torch.Tensor) -> torch.Tensor: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `torch.argmax(logits, dim=-1)` : l'indice du plus grand logit de chaque ligne, de forme `(1,)` ; `[0]` en extrait le tenseur de forme `()`
- Inutile de passer par la softmax : elle ne change pas l'ordre des scores
- `glouton(prompt, 10)` : la fonction décorée prend le prompt et le nombre de tokens à générer

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@boucle_decodage
def glouton(logits: torch.Tensor) -> torch.Tensor:
  return ...


print(glouton("Je suis allé au" if langue == "fr" else "I went to the", 10))
```

### Solution

In [ ]:
@boucle_decodage
def glouton(logits: torch.Tensor) -> torch.Tensor:
  return torch.argmax(logits, dim=-1)[0]


print(glouton("Je suis allé au" if langue == "fr" else "I went to the", 10))

## Test sur des prompts variés

La fonction `teste_decodage` suivante applique une méthode de décodage, avec ses paramètres, à chacun des prompts de la liste `prompts`. Elle affiche les générations et les renvoie dans une liste ; `affiche` les affiche.

*Exécutez les cellules suivantes, qui testent le décodage glouton avec 50 tokens par prompt. Que constatez-vous ?*

In [ ]:
if langue == "fr":
  prompts = [
    "Je suis allé au",
    "Le train pour Lyon",
    "Comment vas-tu",
    "La recette de la tarte aux pommes commence par",
    "Ça va merci, tu devrais",
    "Le chat de la voisine",
    "En 2050, les voitures",
  ]
else:
  prompts = [
    "I went to the",
    "The train to London",
    "How do you",
    "The apple pie recipe starts with",
    "I'm fine thank you, you should",
    "The neighbour's cat",
    "In 2050, cars",
  ]

In [ ]:
def affiche(generations: list[str]) -> None:
  """Affiche chaque prompt de `prompts` avec sa génération."""
  for prompt, generation in zip(prompts, generations):
    print("—" * 80)
    print(f"Prompt : {prompt}")
    print(f"Génération : {generation}")
  print("—" * 80)


def teste_decodage(fonction, *args, **kwargs) -> list[str]:
  """Applique la méthode de décodage `fonction` à chaque prompt de `prompts`."""
  generations = [
    fonction(prompt, *args, **kwargs)
    for prompt in tqdm.auto.tqdm(prompts, desc="Prompts", leave=False)
  ]
  affiche(generations)
  return generations

In [ ]:
generations_gloutonnes = teste_decodage(glouton, 50)

## Mesurer la répétition

Les générations gloutonnes « bouclent » vite : elles répètent les mêmes groupes de mots. Pour comparer les méthodes de décodage sur ce point, mesurons-le par la **part de 3-grammes distincts** d'un texte : on découpe le texte en mots, on forme tous les triplets de mots consécutifs (les 3-grammes), et on divise le nombre de triplets différents par le nombre total de triplets. Elle vaut 1 pour un texte sans répétition, et baisse quand des passages se répètent.

*Codez la fonction `mesure_repetition(texte)` qui renvoie la part de 3-grammes distincts de `texte`. Affichez ensuite sa moyenne sur les générations gloutonnes, `generations_gloutonnes`, avec `statistics.mean`. Les sections suivantes la réutilisent pour chaque méthode.*

In [ ]:
def mesure_repetition(texte: str) -> float: ...


# Votre code ici

### Conseils supplémentaires

Les appels à utiliser :

- `texte.split()` : la liste des mots, séparés par les espaces
- `zip(mots, mots[1:], mots[2:])` : les triplets de mots consécutifs
- `set(...)` : les triplets différents
- `statistics.mean(map(mesure_repetition, generations_gloutonnes))` : la moyenne sur les générations
- Exemple à vérifier : `"a b c a b c"` a 4 triplets, dont 3 différents : 0,75

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def mesure_repetition(texte: str) -> float:
  mots = texte.split()
  trigrammes = list(...)
  return ... / ...


print(mesure_repetition("a b c a b c"))
print(f"Glouton : {statistics.mean(...):.2f}")
```

### Solution

In [ ]:
def mesure_repetition(texte: str) -> float:
  mots = texte.split()
  trigrammes = list(zip(mots, mots[1:], mots[2:]))
  return len(set(trigrammes)) / len(trigrammes)


print(mesure_repetition("a b c a b c"))
print(
  f"Glouton : {statistics.mean(map(mesure_repetition, generations_gloutonnes)):.2f}"
)

Plusieurs générations gloutonnes répètent la même phrase, ou le même groupe de mots, jusqu'à la fin : c'est une faiblesse très classique du décodage glouton. La part de 3-grammes distincts le chiffre : elle est nettement sous 1, là où un texte écrit par un humain en est proche.

## Mesurer la cohérence

La part de 3-grammes distincts mesure la répétition, pas la cohérence : un texte qui n'a plus de sens ne se répète pas non plus. Pour la cohérence, demandons au modèle lui-même. La **log-probabilité moyenne** d'un texte est la moyenne, sur ses tokens, de $\log P(\text{Token}_i | \text{Token}_{j<i})$ selon le modèle : plus elle est proche de 0, plus le modèle trouve le texte probable, et elle baisse quand le texte prend des tours improbables.

Elle se calcule en une seule passe forward sur tout le texte : les logits de la position $i$ donnent la distribution du token $i + 1$, dans laquelle on lit la log-probabilité du token réellement présent.

*Codez la fonction `calcule_log_prob(texte)` qui renvoie la log-probabilité moyenne par token de `texte`, prompt compris. La fonction `mesure(nom, generations)`, fournie, affiche les deux mesures en moyenne sur une liste de générations : appliquez-la aux générations gloutonnes. Les sections suivantes l'utilisent pour chaque méthode.*

**Documentation utile**

- [`torch.nn.functional.log_softmax`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html)
- [`torch.gather`](https://docs.pytorch.org/docs/stable/generated/torch.gather.html)

In [ ]:
def calcule_log_prob(texte: str) -> float: ...


def mesure(nom: str, generations: list[str]) -> None:
  """Affiche les deux mesures, en moyenne sur les générations d'une méthode."""
  repetition = statistics.mean(map(mesure_repetition, generations))
  log_proba = statistics.mean(map(calcule_log_prob, generations))
  print(
    f"{nom} : {repetition:.2f} de 3-grammes distincts, "
    f"log-probabilité moyenne {log_proba:.2f}"
  )


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `ids_tokens = encode(texte)`, puis `modele(ids_tokens).logits`, sans gradients, comme dans « Étude des sorties du modèle pré-entraîné »
- `logits[0, :-1]` : les logits des positions qui ont un token suivant ; `ids_tokens[0, 1:]` : ces tokens suivants
- `torch.nn.functional.log_softmax(..., dim=-1)` : les log-probabilités de tout le vocabulaire, à chaque position
- `log_probas.gather(-1, ids_suivants[:, None])` : à chaque position, la log-probabilité du token suivant réel (`[:, None]` ajoute l'axe que `gather` attend)
- `.mean().item()` : leur moyenne, en nombre Python
- Pour vérifier : `-modele(ids_tokens, labels=ids_tokens).loss` donne la même valeur (le modèle décale lui-même les tokens pour calculer sa perte) ; `calcule_log_prob(prompts[0])`, sur le premier prompt, vaut environ −6,0 avec BelGPT-2 et −3,0 avec GPT-2

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def calcule_log_prob(texte: str) -> float:
  ids_tokens = encode(texte)
  with torch.no_grad():
    logits = modele(ids_tokens).logits
  log_probas = torch.nn.functional.log_softmax(..., dim=-1)
  ids_suivants = ...
  choisies = log_probas.gather(-1, ids_suivants[:, None])
  return ...


print(f"{calcule_log_prob(prompts[0]):.2f}")
mesure("Glouton", generations_gloutonnes)
```

### Solution

In [ ]:
def calcule_log_prob(texte: str) -> float:
  ids_tokens = encode(texte)
  with torch.no_grad():
    logits = modele(ids_tokens).logits
  # La position i prédit le token i + 1 : décalage d'une position
  log_probas = torch.nn.functional.log_softmax(logits[0, :-1], dim=-1)
  ids_suivants = ids_tokens[0, 1:]
  choisies = log_probas.gather(-1, ids_suivants[:, None])
  return choisies.mean().item()


print(f"{calcule_log_prob(prompts[0]):.2f}")
mesure("Glouton", generations_gloutonnes)

Le glouton a une log-probabilité moyenne élevée : il prend à chaque pas le token le plus probable, et une boucle, une fois lancée, devient de plus en plus probable pour le modèle. Une log-probabilité élevée ne suffit donc pas à faire un bon texte : les sections suivantes regardent les deux mesures ensemble.

## Échantillonnage (*sampling*)

Les résultats du décodage glouton bouclent rapidement et sont souvent très génériques. Les méthodes suivantes cherchent à corriger ces défauts.

La première consiste à **échantillonner** à partir des probabilités du modèle, plutôt que de toujours choisir le token le plus probable : un token de probabilité 0,3 est tiré 3 fois sur 10.

*Reprenez la forme de la fonction `glouton`, mais au lieu de choisir l'élément maximal, calculez une distribution de probabilités avec `torch.softmax`, puis tirez le prochain token selon cette distribution avec `torch.multinomial`, dans une fonction `echantillonnage(logits)`. Testez-la avec `teste_decodage` (50 tokens par prompt), et comparez ses deux mesures à celles du décodage glouton.*

**Documentation utile**

- [`torch.softmax`](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html)
- [`torch.multinomial`](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html)

In [ ]:
@boucle_decodage
def echantillonnage(logits: torch.Tensor) -> torch.Tensor: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `torch.softmax(logits, dim=-1)` : les probabilités, de forme `(1, taille du vocabulaire)`
- `torch.multinomial(probabilites, num_samples=1)` : un tirage par ligne, de forme `(1, 1)` ; `[0, 0]` en extrait le tenseur de forme `()`
- `mesure("Échantillonnage", generations)` : les deux mesures

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@boucle_decodage
def echantillonnage(logits: torch.Tensor) -> torch.Tensor:
  probabilites = ...
  return torch.multinomial(...)[0, 0]


generations = teste_decodage(echantillonnage, 50)
mesure("Échantillonnage", generations)
```

### Solution

In [ ]:
@boucle_decodage
def echantillonnage(logits: torch.Tensor) -> torch.Tensor:
  probabilites = torch.softmax(logits, dim=-1)
  return torch.multinomial(probabilites, num_samples=1)[0, 0]


generations = teste_decodage(echantillonnage, 50)
mesure("Échantillonnage", generations)

Les générations ne bouclent plus : la part de 3-grammes distincts est proche de 1. En contrepartie, elles perdent souvent le fil, parce qu'un token peu probable est tiré de temps en temps et que la suite du texte doit s'en accommoder : leur log-probabilité moyenne est nettement plus basse que celle du glouton.

## Température

Avant la softmax, on peut diviser les logits par une **température** $T$ :

- $T < 1$ accentue les écarts : le modèle choisit plus souvent les tokens les plus probables (plus cohérent, moins varié) ;
- $T > 1$ les aplatit : les tokens rares sont plus souvent tirés (plus varié, moins cohérent) ;
- $T \to 0$ revient au décodage glouton.

*Ajoutez un paramètre `temperature` à votre fonction d'échantillonnage, dans une nouvelle fonction `echantillonnage_temperature(logits, temperature)`. Comparez les générations et leurs deux mesures pour $T = 0.5$ et $T = 1.5$.*

In [ ]:
@boucle_decodage
def echantillonnage_temperature(
  logits: torch.Tensor, temperature: float
) -> torch.Tensor: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `logits / temperature` : les logits divisés par la température, avant la softmax
- `echantillonnage_temperature(prompt, 50, temperature=0.5)` : la fonction décorée transmet ses paramètres nommés à la vôtre ; `teste_decodage` aussi

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@boucle_decodage
def echantillonnage_temperature(
  logits: torch.Tensor, temperature: float
) -> torch.Tensor:
  probabilites = torch.softmax(..., dim=-1)
  return torch.multinomial(probabilites, num_samples=1)[0, 0]


for temperature in [0.5, 1.5]:
  generations = teste_decodage(echantillonnage_temperature, 50, temperature=temperature)
  mesure(f"T = {temperature}", generations)
```

### Solution

In [ ]:
@boucle_decodage
def echantillonnage_temperature(
  logits: torch.Tensor, temperature: float
) -> torch.Tensor:
  probabilites = torch.softmax(logits / temperature, dim=-1)
  return torch.multinomial(probabilites, num_samples=1)[0, 0]


for temperature in [0.5, 1.5]:
  generations = teste_decodage(echantillonnage_temperature, 50, temperature=temperature)
  mesure(f"T = {temperature}", generations)

À $T = 0.5$, les générations sont plus cohérentes : leur log-probabilité moyenne se rapproche de celle du glouton ; plus $T$ baisse, plus elles se rapprochent du décodage glouton et de ses boucles. À $T = 1.5$, elles n'ont presque plus de sens : des tokens rares sont tirés à chaque pas, et leur log-probabilité moyenne s'effondre. La part de 3-grammes distincts, elle, sépare beaucoup moins ces deux réglages : elle mesure la répétition, pas la cohérence. La température règle ce compromis entre cohérence et variété, que les deux mesures chiffrent.

## Top-k sampling

Une variation de l'échantillonnage consiste à ne tirer que parmi les `k` tokens les plus probables, pour éviter de générer un token très improbable.

*Repartez de la fonction `echantillonnage` pour coder la fonction `echantillonnage_k(logits, k)` qui implémente cette amélioration avec `torch.topk`. Testez-la avec `teste_decodage`, pour `k = 20`.*

**Documentation utile**

- [`torch.topk`](https://docs.pytorch.org/docs/stable/generated/torch.topk.html)

In [ ]:
@boucle_decodage
def echantillonnage_k(logits: torch.Tensor, k: int) -> torch.Tensor: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `valeurs, indices = torch.topk(logits, k=k, dim=-1)` : les `k` plus grands logits, par ordre décroissant, et leurs indices dans le vocabulaire, tous deux de forme `(1, k)`
- `torch.softmax(valeurs, dim=-1)` : les probabilités des `k` tokens gardés
- `torch.multinomial(probabilites, num_samples=1)[0, 0]` : la position tirée parmi les `k`
- `indices[0, position]` : l'indice dans le vocabulaire du token tiré

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@boucle_decodage
def echantillonnage_k(logits: torch.Tensor, k: int) -> torch.Tensor:
  valeurs, indices = torch.topk(...)
  probabilites = torch.softmax(..., dim=-1)
  tire = torch.multinomial(probabilites, num_samples=1)[0, 0]
  return indices[...]


generations = teste_decodage(echantillonnage_k, 50, k=20)
mesure("Top-k", generations)
```

### Solution

In [ ]:
@boucle_decodage
def echantillonnage_k(logits: torch.Tensor, k: int) -> torch.Tensor:
  valeurs, indices = torch.topk(logits, k=k, dim=-1)
  probabilites = torch.softmax(valeurs, dim=-1)
  tire = torch.multinomial(probabilites, num_samples=1)[0, 0]
  return indices[0, tire]


generations = teste_decodage(echantillonnage_k, 50, k=20)
mesure("Top-k", generations)

## Top-p sampling

Une autre amélioration consiste à ne garder que les tokens les plus probables dont la probabilité cumulée atteint `p` (le *noyau* de la distribution), et pas les suivants. Le nombre de tokens candidats s'adapte alors à la confiance du modèle : peu de candidats quand un token domine, beaucoup quand la distribution est plate.

*Repartez de la fonction `echantillonnage_k` et ajoutez-y le top-p sampling, dans la fonction `echantillonnage_k_p(logits, k, p)`. Vous pourrez utiliser `torch.cumsum`. Testez-la avec `teste_decodage`, pour `k = 20` et `p = 0.85`.*

**Documentation utile**

- [`torch.cumsum`](https://docs.pytorch.org/docs/stable/generated/torch.cumsum.html)

In [ ]:
@boucle_decodage
def echantillonnage_k_p(logits: torch.Tensor, k: int, p: float) -> torch.Tensor: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `torch.softmax(valeurs[0], dim=-1)` : les probabilités des `k` tokens gardés, par ordre décroissant, de forme `(k,)`
- `torch.cumsum(probabilites, dim=0) < p` : vrai pour les tokens avant le seuil ; `.sum()` les compte, et on ajoute 1 pour garder le token qui franchit `p`
- `int(...)` convertit ce nombre en entier Python, pour s'en servir dans une tranche
- Exemple à vérifier : pour les probabilités `[0.5, 0.3, 0.15, 0.05]` et `p = 0.85`, les sommes cumulées sont `[0.5, 0.8, 0.95, 1.0]` : 2 sont sous `p`, on garde 3 tokens
- `torch.multinomial(probabilites[:nb_gardes], num_samples=1)[0]` tire parmi les `nb_gardes` premiers ; `torch.multinomial` n'a pas besoin de probabilités qui somment à 1

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@boucle_decodage
def echantillonnage_k_p(logits: torch.Tensor, k: int, p: float) -> torch.Tensor:
  valeurs, indices = torch.topk(logits, k=k, dim=-1)
  probabilites = ...
  nb_gardes = int((...).sum()) + 1
  tire = torch.multinomial(..., num_samples=1)[0]
  return indices[0, tire]


generations = teste_decodage(echantillonnage_k_p, 50, k=20, p=0.85)
mesure("Top-k et top-p", generations)
```

### Solution

In [ ]:
@boucle_decodage
def echantillonnage_k_p(logits: torch.Tensor, k: int, p: float) -> torch.Tensor:
  valeurs, indices = torch.topk(logits, k=k, dim=-1)
  probabilites = torch.softmax(valeurs[0], dim=-1)
  # Nombre de tokens nécessaires pour que la probabilité cumulée atteigne p
  nb_gardes = int((torch.cumsum(probabilites, dim=0) < p).sum()) + 1
  tire = torch.multinomial(probabilites[:nb_gardes], num_samples=1)[0]
  return indices[0, tire]


generations = teste_decodage(echantillonnage_k_p, 50, k=20, p=0.85)
mesure("Top-k et top-p", generations)

## Génération avec `generate`

En pratique, on n'écrit pas la boucle de décodage : la méthode `generate` des modèles de `transformers` implémente toutes ces stratégies, et d'autres, comme la recherche en faisceau (*beam search*). Elle prend les indices du prompt et un masque d'attention de même forme (1 pour un vrai token, 0 pour une position de remplissage : ici, que des 1), et renvoie les indices du prompt suivis des tokens générés. Ses paramètres règlent le choix du token suivant : `do_sample=False` pour le décodage glouton, `do_sample=True` pour l'échantillonnage, avec `temperature`, `top_k` et `top_p`.

*Codez la fonction `genere_transformers(prompt, longueur, **reglages)`, qui génère `longueur` tokens après `prompt` avec `modele.generate`, en lui passant les réglages `reglages`, et renvoie le texte. Testez-la avec `teste_decodage`, avec une centaine de tokens par prompt : d'abord avec un top-p sampling restreint aux 30 tokens les plus probables, de noyau `p = 0.95`, puis avec le décodage glouton. Comparez leurs deux mesures.*

**Documentation utile**

- [`generate`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.GenerationMixin.generate)
- [`GenerationConfig`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.GenerationConfig)

In [ ]:
def genere_transformers(prompt: str, longueur: int, **reglages) -> str: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `ids_tokens = encode(prompt)`, et `torch.ones_like(ids_tokens)` pour le masque d'attention (`attention_mask`)
- `max_new_tokens=longueur` : le nombre de tokens à générer, sans compter le prompt
- `max_length=None` : le `config.json` de BelGPT-2 (qui n'a pas de `generation_config.json`) fixe `max_length` à 20, la valeur par défaut de `transformers`, enregistrée avec le modèle : 20 tokens au plus, prompt compris ; on retire cette limite, pour que seul `max_new_tokens` compte (sinon, `transformers` avertit que les deux limites sont fixées)
- `pad_token_id=id_token_debut, eos_token_id=id_token_debut` : les symboles de remplissage et de fin, que la configuration de BelGPT-2 donne faux (voir « Inspection du modèle »)
- `**reglages` transmet les réglages : `{"do_sample": True, "top_k": 30, "top_p": 0.95}` pour le top-p sampling, `{"do_sample": False}` pour le décodage glouton
- `decode(ids_generes)` : `generate` renvoie un tenseur de forme `(1, tokens du prompt + longueur)`, qui commence par le prompt

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def genere_transformers(prompt: str, longueur: int, **reglages) -> str:
  ids_tokens = encode(prompt)
  ids_generes = modele.generate(
    ids_tokens,
    attention_mask=...,
    max_new_tokens=...,
    max_length=None,
    pad_token_id=id_token_debut,
    eos_token_id=id_token_debut,
    **reglages,
  )
  return ...


for reglages in [{"do_sample": True, "top_k": 30, "top_p": 0.95}, {"do_sample": False}]:
  generations = teste_decodage(genere_transformers, 100, **reglages)
  mesure(str(reglages), generations)
```

### Solution

In [ ]:
def genere_transformers(prompt: str, longueur: int, **reglages) -> str:
  ids_tokens = encode(prompt)
  ids_generes = modele.generate(
    ids_tokens,
    attention_mask=torch.ones_like(ids_tokens),
    max_new_tokens=longueur,
    max_length=None,
    pad_token_id=id_token_debut,
    eos_token_id=id_token_debut,
    **reglages,
  )
  return decode(ids_generes)


for reglages in [{"do_sample": True, "top_k": 30, "top_p": 0.95}, {"do_sample": False}]:
  generations = teste_decodage(genere_transformers, 100, **reglages)
  mesure(str(reglages), generations)

Sur une centaine de tokens, l'écart se creuse : la plupart des générations gloutonnes finissent par boucler, et leur part de 3-grammes distincts chute, alors que le top-p sampling reste varié. Le glouton garde pourtant la log-probabilité moyenne la plus haute : ses boucles sont très probables pour le modèle. Aucune des deux mesures ne suffit seule. C'est pourquoi les modèles de langage sont presque toujours décodés par échantillonnage, avec une température, un top-k et un top-p réglés pour l'usage visé.